testin scripts/train.py

In [ ]:
config_dir = "/Users/sophiali/Desktop/chip-stroma-analysis/configs/local"

In [ ]:
import argparse as ap
import sys
import os

from pathlib import Path
from datetime import datetime

project_root = Path.cwd().parent
sys.path.append(str(project_root / "src"))

os.chdir(Path.cwd().parent)

from chip_stroma.utils.config import load_configs
from chip_stroma.utils.loggers import setup_logger
from chip_stroma.utils.io import initialize_train_manifest
from chip_stroma.training.train import train

logger = setup_logger(__name__)

%load_ext autoreload
%autoreload 2

In [ ]:
def log_header(config_path):
    logger.info("=" * 60)
    logger.info("Starting Pipeline Execution")
    logger.info("- Pipeline Stage: Segmentation UNet Training")
    logger.info(f"- Configurations: {config_path}")
    logger.info(f"- Working Directory: {Path.cwd()}")
    logger.info(f"- Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}")
    logger.info("=" * 60)

def log_footer(cfg, metrics):
    logger.info("=" * 60)
    logger.info("Successfully Completed Pipeline Execution")
    logger.info(f"- Train Manifest: {cfg.metadata.train_manifest}")
    logger.info(f"- Validation Loss: {metrics.get('val/loss')}")
    logger.info(f"- Validation Dice Score: {metrics.get('val/dice')}")
    logger.info(f"- Validation IoU Score: {metrics.get('val/iou')}")
    logger.info(f"- Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}")
    logger.info("=" * 60)

In [ ]:
log_header(config_path = Path(config_dir) / "segmentation.yaml")

# 1. Load workflow and path configurations
config = load_configs(
    pipeline = Path(config_dir) / "segmentation.yaml",
    paths    = Path(config_dir) / "paths.yaml"
)

In [ ]:
# 2. Verify that the training manifest was created, else create it
manifest = initialize_train_manifest(
    train_manifest_path = config.paths.metadata.train_manifest,
    patch_manifest_path = config.paths.metadata.patch_manifest
)

In [ ]:
# Train the model using the configurations provided in the YAML
metrics = train(
    manifest = manifest,
    project  = config.segmentation.study.project,
    group    = config.segmentation.study.group,
    paths    = config.paths,
    params   = config.segmentation,
    seed     = config.segmentation.data.seed,
    trial    = None
)

In [ ]:
log_footer(cfg = config.paths, metrics = metrics)